In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
## Loading in datasets

itu = pd.read_csv(
    "../data/interim/itu_final_candidates_2018_2024.csv"
)

electricity = pd.read_csv(
    "../data/raw/API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_367283.csv",
    skiprows=4
)

classifications = pd.read_csv(
    "../data/raw/nations_incomes_2026_07_15.csv"
)

In [4]:
print(itu.shape)
print(electricity.shape)
print(classifications.shape)

(1016, 15)
(265, 71)
(268, 5)


In [6]:
final_temp_countries = itu["AreaName"].unique()

final_temp_countries

array(['Afghanistan', 'Angola', 'Armenia', 'Burkina Faso', 'Botswana',
       'Barbados', 'Cambodia', 'Colombia', 'Spain', 'Ethiopia',
       'Honduras', 'India', 'Indonesia', 'Iraq', 'Israel', 'Japan',
       'Jamaica', 'Kenya', 'Madagascar', 'Morocco', 'Papua New Guinea',
       'Rwanda', 'Uganda', 'Ukraine', 'United States'], dtype=object)

In [7]:
len(final_temp_countries)

25

In [8]:
electricity_final = electricity[electricity["Country Name"].isin(final_temp_countries)].copy()

electricity_final["Country Name"].unique()

array(['Afghanistan', 'Angola', 'Armenia', 'Burkina Faso', 'Barbados',
       'Botswana', 'Colombia', 'Spain', 'Ethiopia', 'Honduras',
       'Indonesia', 'India', 'Iraq', 'Israel', 'Jamaica', 'Japan',
       'Kenya', 'Cambodia', 'Morocco', 'Madagascar', 'Papua New Guinea',
       'Rwanda', 'Uganda', 'Ukraine', 'United States'], dtype=object)

In [9]:
electricity_final["Country Name"].nunique()

25

In [11]:
# Melting the electricity dataframe into a better form for conducting the analysis

year_cols = [str(year) for year in range(2018,2025)]

electricity_melted_final = electricity_final.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars= year_cols,
    var_name= "Year",
    value_name="Electricity_Access"
)

In [12]:
electricity_melted_final["Year"] = electricity_melted_final["Year"].astype(int)

In [13]:
electricity_melted_final.shape

(175, 4)

In [14]:
electricity_melted_final.head(10)

,Country Name,Country Code,Year,Electricity_Access
0,Afghanistan,AFG,2018,93.4
1,Angola,AGO,2018,45.3
2,Armenia,ARM,2018,99.9
3,Burkina Faso,BFA,2018,14.4
4,Barbados,BRB,2018,100.0
5,Botswana,BWA,2018,68.3
6,Colombia,COL,2018,98.5
7,Spain,ESP,2018,100.0
8,Ethiopia,ETH,2018,44.8
9,Honduras,HND,2018,91.6


In [17]:
# Transforming the world income data

classifications_final = classifications[["Code", "Region", "Income group"]].copy()

In [19]:
electricity_merged = electricity_melted_final.merge(
    classifications_final,
    left_on="Country Code",
    right_on="Code",
    how="left"
)

In [20]:
electricity_merged.shape

(175, 7)

In [21]:
electricity_merged.head(10)

,Country Name,Country Code,Year,Electricity_Access,Code,Region,Income group
0,Afghanistan,AFG,2018,93.4,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,Angola,AGO,2018,45.3,AGO,Sub-Saharan Africa,Lower middle income
2,Armenia,ARM,2018,99.9,ARM,Europe & Central Asia,Upper middle income
3,Burkina Faso,BFA,2018,14.4,BFA,Sub-Saharan Africa,Low income
4,Barbados,BRB,2018,100.0,BRB,Latin America & Caribbean,High income
5,Botswana,BWA,2018,68.3,BWA,Sub-Saharan Africa,Upper middle income
6,Colombia,COL,2018,98.5,COL,Latin America & Caribbean,Upper middle income
7,Spain,ESP,2018,100.0,ESP,Europe & Central Asia,High income
8,Ethiopia,ETH,2018,44.8,ETH,Sub-Saharan Africa,Low income
9,Honduras,HND,2018,91.6,HND,Latin America & Caribbean,Lower middle income


In [22]:
electricity_merged = electricity_merged.drop(columns=["Code"])

In [23]:
electricity_merged.head(10)

,Country Name,Country Code,Year,Electricity_Access,Region,Income group
0,Afghanistan,AFG,2018,93.4,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,Angola,AGO,2018,45.3,Sub-Saharan Africa,Lower middle income
2,Armenia,ARM,2018,99.9,Europe & Central Asia,Upper middle income
3,Burkina Faso,BFA,2018,14.4,Sub-Saharan Africa,Low income
4,Barbados,BRB,2018,100.0,Latin America & Caribbean,High income
5,Botswana,BWA,2018,68.3,Sub-Saharan Africa,Upper middle income
6,Colombia,COL,2018,98.5,Latin America & Caribbean,Upper middle income
7,Spain,ESP,2018,100.0,Europe & Central Asia,High income
8,Ethiopia,ETH,2018,44.8,Sub-Saharan Africa,Low income
9,Honduras,HND,2018,91.6,Latin America & Caribbean,Lower middle income


In [24]:
# Merging ITU data into the frame to have a complete dataset to analyze 
itu_wide = itu.pivot(
    index=["AreaName", "AreaCode", "Year"],
    columns="Series",
    values="Value"
).reset_index()

In [26]:
print(itu_wide.shape)
itu_wide.head(10)

(175, 9)


Series,AreaName,AreaCode,Year,Active mobile-broadband subscriptions,Estimated proportion of households with Internet access at home,Fixed-broadband subscriptions,Internet users (%),Percentage of the population covered by at least an LTE/WiMAX mobile network.,Population
0,Afghanistan,AFG,2018,6997208.0,7.270304,15999.0,16.799999,7.000000,36743039.0
1,Afghanistan,AFG,2019,7309797.0,8.438115,19683.0,17.600000,22.000000,37856121.0
2,Afghanistan,AFG,2020,7422550.0,NaN,26570.0,17.048500,26.000000,39068979.0
3,Afghanistan,AFG,2021,7422550.0,24.288903,26570.0,16.514299,26.000000,40000412.0
4,Afghanistan,AFG,2022,22831896.0,27.592028,32313.0,15.866300,33.030000,40578842.0
5,Afghanistan,AFG,2023,23027082.0,27.700000,33200.0,15.928400,34.000000,41454761.0
6,Afghanistan,AFG,2024,25630751.0,27.989760,NaN,16.094999,37.324101,42647492.0
7,Angola,AGO,2018,5820154.0,6.711401,109561.0,29.000000,8.000000,31297155.0
8,Angola,AGO,2019,6740418.0,8.016441,119068.0,32.129398,18.000000,32375632.0
9,Angola,AGO,2020,6637340.0,35.390813,121516.0,33.320900,72.740000,33451132.0


In [27]:
analysis_df = itu_wide.merge(
    electricity_merged,
    left_on=["AreaCode", "Year"],
    right_on=["Country Code", "Year"],
    how="left"
)

In [28]:
print(analysis_df.shape)
analysis_df.head()

(175, 14)


,AreaName,AreaCode,Year,Active mobile-broadband subscriptions,Estimated proportion of households with Internet access at home,Fixed-broadband subscriptions,Internet users (%),Percentage of the population covered by at least an LTE/WiMAX mobile network.,Population,Country Name,Country Code,Electricity_Access,Region,Income group
0,Afghanistan,AFG,2018,6997208.0,7.270304,15999.0,16.799999,7.00,36743039.0,Afghanistan,AFG,93.4,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,Afghanistan,AFG,2019,7309797.0,8.438115,19683.0,17.600000,22.00,37856121.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income
2,Afghanistan,AFG,2020,7422550.0,NaN,26570.0,17.048500,26.00,39068979.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income
3,Afghanistan,AFG,2021,7422550.0,24.288903,26570.0,16.514299,26.00,40000412.0,Afghanistan,AFG,97.7,"Middle East, North Africa, Afghanistan & Pakistan",Low income
4,Afghanistan,AFG,2022,22831896.0,27.592028,32313.0,15.866300,33.03,40578842.0,Afghanistan,AFG,85.3,"Middle East, North Africa, Afghanistan & Pakistan",Low income


In [29]:
analysis_df["Electricity_Access"].isna().sum()

0